# spVIPESmulti: Normalizing Flow vs Gaussian Prior on Perturbation Data

This vignette demonstrates how **spVIPESmulti** decomposes single-cell perturbation
data into **shared** (cell-type identity) and **private** (perturbation-specific)
latent representations. We compare the standard Gaussian prior against a
**Neural Spline Flow (NSF)** prior on the shared latent space.

**Dataset.** IFN-β stimulated PBMCs from the CINEMA-OT example
(Dong *et al.*, *Nature Methods* 2023), originally derived from
Kang *et al.* (2018). The dataset contains ~1 000 cells split between
an unstimulated control and IFN-β treatment, with annotated cell types.

**Goal.** The shared latent should capture cell-type identity (invariant
across conditions), while the private latent should capture
perturbation-specific responses. A normalizing-flow prior can model
multi-modal latent distributions more faithfully than a standard Gaussian,
potentially improving disentanglement.

| Step | What we do |
|---|---|
| §1 | Load the CinemaOT example dataset |
| §2 | Run CinemaOT as a reference decomposition |
| §3 | Prepare data for spVIPESmulti |
| §4 | Train spVIPESmulti with **Gaussian** prior (baseline) |
| §5 | Train spVIPESmulti with **NSF** prior |
| §6 | Compare shared latent spaces (UMAP) |
| §7 | Compare private latent spaces (UMAP) |
| §8 | Quantitative evaluation |

## Requirements & Compatibility

> **scvi-tools ≥1.0 required.** spVIPESmulti targets scvi-tools 1.x and `lightning.pytorch` (was 0.20.x + `pytorch_lightning`). The deprecated `use_gpu=True` kwarg on `model.train(...)` has been **removed**; pass `accelerator="gpu", devices=1` (or `"auto"`) inside `**trainer_kwargs` instead. Minimum Python is 3.10. Several private scvi-tools modules removed in 1.x are now vendored under `spVIPESmulti.data`. See `CHANGELOG.md` for full details.

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import torch

import pertpy as pt
import spVIPESmulti

np.random.seed(42)
torch.manual_seed(42)
sc.settings.set_figure_params(dpi=100, frameon=False)
torch.cuda.is_available()

ImportError: cannot import name 'xla_pmap_p' from 'jax.extend.core.primitives' (/exports/archive/hg-funcgenom-research/mdmanurung/conda/envs/scvi-test/lib/python3.13/site-packages/jax/extend/core/primitives.py)

## 1. Load the CinemaOT example dataset

`pt.dt.cinemaot_example()` returns ~1 000 PBMCs with two perturbation
conditions (`"No stimulation"` and `"IFNb"`) and cell-type annotations
in `cell_type0528`.

In [ ]:
adata = pt.dt.cinemaot_example()

print(f"Shape           : {adata.shape}")
print(f"Perturbation    : {adata.obs['perturbation'].value_counts().to_dict()}")
print(f"Cell types      : {adata.obs['cell_type0528'].nunique()}")
print()
print(adata.obs['cell_type0528'].value_counts())

In [ ]:
sc.pl.umap(adata, color=["perturbation", "cell_type0528"], wspace=0.5)

## 2. CinemaOT reference decomposition

CINEMA-OT uses optimal transport to separate **confounder** variation
(cell type) from **treatment effects**. We run it here as a reference
point: the confounder embedding is analogous to spVIPESmulti' shared latent,
and the treatment-effect embedding is analogous to the private latent.

In [ ]:
cot = pt.tl.Cinemaot()
sc.pp.pca(adata)
de = cot.causaleffect(
    adata,
    pert_key="perturbation",
    control="No stimulation",
    return_matching=True,
    thres=0.5,
    smoothness=1e-5,
    eps=1e-3,
    solver="Sinkhorn",
    preweight_label="cell_type0528",
)

In [ ]:
# Confounder embedding (analogous to spVIPESmulti shared latent)
sc.pp.neighbors(adata, use_rep="cf")
sc.tl.umap(adata, random_state=0)
adata.obsm["X_umap_cinemaot_cf"] = adata.obsm["X_umap"].copy()

# Treatment-effect embedding (analogous to spVIPESmulti private latent)
sc.pp.neighbors(de, use_rep="X_embedding")
sc.tl.umap(de, random_state=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sc.pl.embedding(adata, basis="umap_cinemaot_cf", color="cell_type0528",
                ax=axes[0], show=False, title="CinemaOT confounder — cell type")
sc.pl.embedding(adata, basis="umap_cinemaot_cf", color="perturbation",
                ax=axes[1], show=False, title="CinemaOT confounder — perturbation")
sc.pl.umap(de, color="cell_type0528",
           ax=axes[2], show=False, title="CinemaOT treatment effect")
plt.tight_layout()
plt.show()

In the confounder space, treated and control cells overlap (confounders
removed), while cell types remain separated — confirming that CINEMA-OT
successfully isolates treatment effects from biological identity.

We now replicate this decomposition with spVIPESmulti and compare the
Gaussian vs NSF prior.

## 3. Prepare data for spVIPESmulti

We split the dataset by `perturbation` into two groups and feed them
to `prepare_adatas`. spVIPESmulti will learn a **shared** latent (capturing
cell-type identity across both conditions) and two **private** latents
(capturing condition-specific variation).

In [ ]:
control = adata[adata.obs["perturbation"] == "No stimulation"].copy()
treated = adata[adata.obs["perturbation"] == "IFNb"].copy()

print(f"Control : {control.shape}")
print(f"Treated : {treated.shape}")

In [ ]:
sdata = spVIPESmulti.data.prepare_adatas({
    "control": control,
    "IFNb": treated,
})

print(f"Combined shape : {sdata.shape}")
print(f"Groups         : {sdata.obs['groups'].value_counts().to_dict()}")

In [ ]:
spVIPESmulti.model.spVIPESmulti.setup_anndata(
    sdata,
    groups_key="groups",
    label_key="cell_type0528",
)

group_indices_list = [
    np.where(sdata.obs["groups"] == g)[0]
    for g in sdata.obs["groups"].unique()
]
print(f"Cells per group: {[len(g) for g in group_indices_list]}")

## 4. Baseline — Gaussian prior

We first train a model with the standard $\mathcal{N}(0, I)$ prior on
both latent spaces.

In [ ]:
N_SHARED   = 10
N_PRIVATE  = 5
N_HIDDEN   = 128
DROPOUT    = 0.1
MAX_EPOCHS = 80
BATCH_SIZE = 128
KL_WARMUP  = 30

torch.manual_seed(42)
np.random.seed(42)

model_gauss = spVIPESmulti.model.spVIPESmulti(
    sdata,
    n_hidden=N_HIDDEN,
    n_dimensions_shared=N_SHARED,
    n_dimensions_private=N_PRIVATE,
    dropout_rate=DROPOUT,
    use_nf_prior=False,
)
print(model_gauss)

In [ ]:
model_gauss.train(
    group_indices_list,
    max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    train_size=0.9,
    early_stopping=False,
    n_epochs_kl_warmup=KL_WARMUP,
)

In [ ]:
spVIPESmulti.pl.training_curves(model_gauss)

## 5. Neural Spline Flow prior

We now train an identical architecture but replace the Gaussian prior on
the **shared** latent with a Neural Spline Flow (NSF). The NSF can model
multi-modal and skewed distributions, which may better capture the
structure of the shared representation.

Under the hood the KL term becomes a Monte Carlo estimate:

$$
\mathrm{KL}\bigl(q(z \mid x) \,\|\, p_\text{flow}(z)\bigr)
\;\approx\;
\log q(z \mid x) - \log p_\text{flow}(z)
$$

The flow parameters are jointly optimised with the VAE — no extra
warm-up schedule is needed.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

model_nf = spVIPESmulti.model.spVIPESmulti(
    sdata,
    n_hidden=N_HIDDEN,
    n_dimensions_shared=N_SHARED,
    n_dimensions_private=N_PRIVATE,
    dropout_rate=DROPOUT,
    use_nf_prior=True,
    nf_type="NSF",
    nf_transforms=3,
    nf_target="shared",
)
print(model_nf)

In [ ]:
model_nf.train(
    group_indices_list,
    max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    train_size=0.9,
    early_stopping=False,
    n_epochs_kl_warmup=KL_WARMUP,
)

In [ ]:
spVIPESmulti.pl.training_curves(model_nf)

## 5b. NSF prior + Disentanglement objective

spVIPESmulti supports an optional **disentanglement objective** (inspired by
CellDISECT and Multi-ContrastiveVAE) that enforces:

* `z_shared` encodes cell-type biology but **not** perturbation status
* `z_private` encodes perturbation-specific variation but **not** cell-type biology

We train a third model with the NSF prior **and** the full disentanglement
preset, so we can ask whether explicit disentanglement adds value on top of
an already-flexible flow prior. See `disentangle_ablation.ipynb` for a
per-component ablation.


In [ ]:
torch.manual_seed(42)
np.random.seed(42)

model_nf_dis = spVIPESmulti.model.spVIPESmulti(
    sdata,
    n_hidden=N_HIDDEN,
    n_dimensions_shared=N_SHARED,
    n_dimensions_private=N_PRIVATE,
    dropout_rate=DROPOUT,
    use_nf_prior=True,
    nf_type="NSF",
    nf_transforms=3,
    nf_target="shared",
    disentangle_preset="full",
)
model_nf_dis.train(
    group_indices_list,
    max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    train_size=0.9,
    early_stopping=False,
    n_epochs_kl_warmup=KL_WARMUP,
)


## 6. Compare shared latent spaces

The **shared** latent should capture cell-type identity while mixing
cells from both perturbation conditions. Good integration means:

- Cell types are well-separated (high biological conservation)
- Perturbation conditions are well-mixed (high batch mixing)

In [ ]:
latents_gauss = model_gauss.get_latent_representation(group_indices_list, batch_size=BATCH_SIZE)
latents_nf = model_nf.get_latent_representation(group_indices_list, batch_size=BATCH_SIZE)
latents_nf_dis = model_nf_dis.get_latent_representation(group_indices_list, batch_size=BATCH_SIZE)

print("Latent keys:", list(latents_gauss.keys()))


In [ ]:
# Store shared latents for all three models; snapshot under model-specific keys
for latents_x, rep_key, umap_key in [
    (latents_gauss,   "X_shared_gauss",  "umap_gauss"),
    (latents_nf,      "X_shared_nf",     "umap_nf"),
    (latents_nf_dis,  "X_shared_nf_dis", "umap_nf_dis"),
]:
    spVIPESmulti.utils.store_latents(sdata, latents_x, group_indices_list)
    sdata.obsm[rep_key] = sdata.obsm.pop("X_spVIPESmulti_shared")
    spVIPESmulti.utils.compute_shared_umap(sdata, obsm_key=rep_key, umap_key=umap_key)


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 14))
for row, (key_added, title) in enumerate([
    ("gauss", "Gaussian"),
    ("nf", "NSF"),
    ("nf_dis", "NSF + disentangle=full"),
]):
    sc.pl.embedding(sdata, basis=f"umap_{key_added}", color="cell_type0528",
                    ax=axes[row, 0], show=False,
                    title=f"{title} shared — cell type")
    sc.pl.embedding(sdata, basis=f"umap_{key_added}", color="perturbation",
                    ax=axes[row, 1], show=False,
                    title=f"{title} shared — perturbation")
plt.tight_layout(); plt.show()


## 7. Compare private latent spaces

The **private** latent for the IFN-β group should capture
perturbation-specific gene programs — variation that is *not* shared
with the unstimulated control. Ideally, cell-type structure should be
absent (or much weaker) in the private space, since cell identity is
a shared confounder.

In [ ]:
groups = list(sdata.obs["groups"].unique())

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for row, (latents, label) in enumerate([(latents_gauss, "Gaussian"),
                                         (latents_nf, "NSF")]):
    for col, gi in enumerate(range(len(groups))):
        g_positions = sdata.uns["groups_obs_indices"][gi]
        g_obs = sdata.obs.iloc[g_positions].copy()

        priv = ad.AnnData(
            X=latents["private_reordered"][gi],
            obs=g_obs,
        )
        sc.pp.neighbors(priv, use_rep="X", n_neighbors=15)
        sc.tl.umap(priv, random_state=0)

        sc.pl.umap(priv, color="cell_type0528", ax=axes[row, col],
                   show=False,
                   title=f"{label} private — {groups[gi]}")

plt.tight_layout()
plt.show()

If the model successfully disentangles, the private UMAPs should show
**weaker** cell-type clustering than the shared UMAPs — cell identity is
pushed to the shared space, while perturbation-specific variation stays
private.

## 8. Quantitative comparison

We report two complementary metrics on the **shared** latent:

| Metric | Measures | Good value |
|---|---|---|
| **ARI** (Adjusted Rand Index) | Biological conservation — do Leiden clusters match cell types? | Higher ↑ |
| **Batch mixing entropy** | Perturbation mixing — are conditions mixed in local neighbourhoods? | Higher ↑ |

A successful shared latent achieves **high ARI and high batch mixing
simultaneously**.

In [ ]:
cell_types = sdata.obs["cell_type0528"].values
perturbation = sdata.obs["groups"].values

results = []
for name, key in [("Gaussian", "X_shared_gauss"), ("NSF", "X_shared_nf")]:
    rep = sdata.obsm[key]
    ari = spVIPESmulti.metrics.leiden_ari(rep, cell_types)
    mix = spVIPESmulti.metrics.ilisi(rep, perturbation, k=30)
    results.append({"prior": name, "ARI (cell types)": ari, "batch mixing": mix})

results_df = pd.DataFrame(results).set_index("prior")
results_df.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(results_df))
width = 0.35
ax.bar(x - width / 2, results_df["ARI (cell types)"],  width, label="ARI (cell types) ↑")
ax.bar(x + width / 2, results_df["batch mixing"],      width, label="batch mixing ↑")
ax.set_xticks(x)
ax.set_xticklabels(results_df.index)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right", fontsize=9)
ax.set_title("Shared-latent quality: Gaussian vs NSF prior")
plt.tight_layout()
plt.show()

## Summary

We used the CINEMA-OT perturbation dataset to compare two spVIPESmulti
configurations on the task of disentangling **cell identity** (shared)
from **IFN-β treatment effects** (private).

| | Gaussian prior | NSF prior |
|---|---|---|
| **Shared latent prior** | $\mathcal{N}(0, I)$ | Neural Spline Flow |
| **KL computation** | Closed-form | Monte Carlo |
| **Extra parameters** | — | Flow transforms |

**Key takeaways:**

- Both models successfully decompose the data into shared (cell type)
  and private (perturbation) latent spaces, comparable to the CinemaOT
  confounder / treatment-effect decomposition.
- The NSF prior gives the shared latent more flexibility to model
  non-Gaussian structure, which can improve biological conservation
  and batch mixing — especially on larger datasets and higher
  dimensional latent spaces.
- For a thorough benchmark, scale up to the full dataset and increase
  `MAX_EPOCHS`.

### References

- Novella-Rausell, C., Peters, D.J.M., & Mahfouz, A. (2023).
  *Integrative learning of disentangled representations in multi-view
  single-cell data.* bioRxiv.
- Dong, M. *et al.* (2023). *Causal identification of single-cell
  experimental perturbation effects with CINEMA-OT.* Nature Methods.
- Durkan, C. *et al.* (2019). *Neural Spline Flows.* NeurIPS.